In [ ]:
with open(token_dict_path, 'wb') as f:
    pickle.dump(token_dict, f)

In [ ]:
import json

In [ ]:
json_file = '/lustre/scratch126/cellgen/team361/data/sanger_whole_embryo/xenium/section020_SOB26/output-XETG00155__0004457__SOB26-WEM-FFPE-1-S20__20240912__123924/gene_panel.json'

In [ ]:
with open(json_file, 'r') as file:
    gene_panel = json.load(file)

In [ ]:
gene_panel.keys()

In [ ]:
len(gene_panel['payload']['targets'])

In [ ]:
gene_panel['metadata']

In [ ]:
import scanpy as sc
adata = sc.read_h5ad(/adata_batch0.h5ad')

In [ ]:
import os

In [ ]:
for subdir, _, files in os.walk('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(file_path)
            print(adata.obs['cell_id'].values[0])
            #print(adata.obs['sample'].unique())
            print(adata.uns['batch'])
            print("================")

In [ ]:
adata.X.toarray()

In [ ]:
import numpy as np
adata.X = adata.X.astype(np.float64)

In [ ]:
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=False)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color='niche_name')

In [ ]:
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.X = sc.pp.log1p(scales_counts["X"], copy=True)
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=False)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color='niche_name')

In [ ]:
sc.pl.spatial(adata,
              color='niche_name',
              spot_size=30)

In [ ]:
adata.obs['niche_name'].value_counts()

In [ ]:
adata.obsm['spatial_connectivities']

In [ ]:
# Load hold out skin samples and concatenate in one adata object
adata_list = []

for subdir, _, files in os.walk('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(adata.X)
            adata.X = adata.X.astype(int64)
            print(adata.X)

adata = ad.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata2 = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_1p/harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad')

In [ ]:
import pickle

# Path to your pickle file
pickle_file_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/human_cohort1_10m/token_dictionary.pkl'

# Load the pickle file
with open(pickle_file_path, 'rb') as file:
    token_dict = pickle.load(file)

# Now `token_dict` is a Python dictionary
print(token_dict)

In [ ]:
spv_count = sum(1 for key in token_dict if "spv" in key)
spv_count

In [ ]:
len(token_dict)

In [ ]:
import os
import pickle

import scanpy as sc
import squidpy as sq

In [ ]:
folder_path = '/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p'
token_dict_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-neighborhood-tokenizer/xhs1000-39b_1p/token_dictionary.pkl'

In [ ]:
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

In [ ]:
len(token_dict)

In [ ]:
count_spv_keys = sum(1 for key in token_dict if key.startswith("spv"))
count_spv_keys

In [ ]:
token_dict[list(token_dict.keys())[-1:][0]]

In [ ]:
token_dict['ENSG00000292373']

In [ ]:
max_nonzero_counts = 0  # Initialize a variable to store the maximum count
neighbor_max_nonzero_counts = 0

# Iterate through all .h5ad files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.h5ad'):
        file_path = os.path.join(folder_path, filename)
        
        # Load the .h5ad file
        adata = sc.read_h5ad(file_path)

        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                delaunay=True,
                                set_diag=True)

        adata.layers['X_neighborhood'] = (
                adata.obsp['spatial_connectivities'].T @ adata.X)
        
        # Calculate nonzero counts per cell
        nonzero_counts = (adata.X > 0).sum(axis=1)
        neighbor_nonzero_counts = (adata.layers['X_neighborhood'] > 0).sum(axis=1)
        
        # Get the maximum nonzero count in the current file
        file_max_nonzero = nonzero_counts.max()
        file_max_neighbor_nonzero = neighbor_nonzero_counts.max()
        
        # Update the overall maximum if the current file has a higher value
        if file_max_nonzero > max_nonzero_counts:
            max_nonzero_counts = file_max_nonzero

        if file_max_neighbor_nonzero > neighbor_max_nonzero_counts:
            neighbor_max_nonzero_counts = file_max_neighbor_nonzero

In [ ]:
adata.obs

In [ ]:
max_nonzero_counts

In [ ]:
neighbor_max_nonzero_counts

In [ ]:
adata_test = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhs1000-1b_1p/adata_5k_xenium_skin.h5ad')

In [ ]:
adata_test.obs['info_id']

In [ ]:
batch_mapping_dict = {value: idx for idx, value in enumerate(adata_test.obs['info_id'].unique().tolist())}

In [ ]:
batch_mapping_dict # 19, 11, 32

In [ ]:
2+3

In [ ]:
'/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhe1009-20b_1p/section056_SOB26/output-XETG00155__0003607__SOB25-WEM-FFPE-1-S56__20240912__123924.zip'

In [ ]:
chmod -R u+rw /lustre/scratch126/cellgen/team361/DATASETS/bronze/xhe1009-20b_1p
